In [4]:
!pip install requests
!pip install json
!pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement json (from versions: none)

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for json



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import requests
import pandas as pd
import time

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

TOTW_URL = "https://www.fotmob.com/api/data/team-of-the-week/team"

ligas_totw = {
    "Premier League": {"id": 47, "rondas": 38},
    "LaLiga":         {"id": 87, "rondas": 38},
    "Serie A":        {"id": 55, "rondas": 38},
    "Bundesliga":     {"id": 54, "rondas": 34},
    "Ligue 1":        {"id": 53, "rondas": 34},
}


def obtener_totw_jornada(id_liga, round_id, season="2025/2026", mapa_equipos=None):
    """Descarga el Equipo de la Semana de UNA jornada concreta."""
    params = {"leagueId": id_liga, "roundId": round_id, "season": season}
    r = requests.get(TOTW_URL, params=params)
    if r.status_code != 200:
        return None

    data = r.json()
    if not isinstance(data, list) or not data:
        return None
    filas = []
    for jugador in data:
        nombre = jugador.get("name", {}) or {}
        rating = jugador.get("rating", {}) or {}
        layout = jugador.get("verticalLayout", {}) or {}
        team_id = jugador.get("teamId")
        is_top = jugador.get("isTopPlayer", {}) or {}

        filas.append({
            "player_id": jugador.get("id"),
            "player": nombre.get("fullName"),
            "rating": rating.get("num"),
            "rating.bgcolor": rating.get("bgcolor"),
            "teamId": team_id,
            "team": (mapa_equipos or {}).get(team_id),
            "pos_x": layout.get("x"),
            "pos_y": layout.get("y"),
            "matchId": jugador.get("matchId"),
            "isTopPlayer": is_top.get("isTopPlayer"),
            "isTots": jugador.get("isTots"),
            "Jornada": round_id,
        })

    df = pd.DataFrame(filas)
    df = df.drop(columns="isTots", errors="ignore")
    df["Temporada"] = season
    return df


def mapa_equipos_liga(id_liga, season="2025/2026", pais="ESP"):
    """
    Usa la tabla de clasificación (que ya sabemos leer bien) para
    construir un diccionario {teamId: nombre_equipo}.
    """
    params = {"id": id_liga, "ccode3": pais, "season": season}
    r = requests.get("https://www.fotmob.com/api/data/leagues", params=params)
    r.raise_for_status()
    data = r.json()

    tablas = data.get("table", [])
    for bloque in tablas:
        tabla_obj = bloque.get("data", {}).get("table", {})
        filas_all = tabla_obj.get("all")
        if filas_all:
            return {f["id"]: f["name"] for f in filas_all if "id" in f and "name" in f}
    return {}


def obtener_totw_temporada(nombre_liga, id_liga, num_rondas, season="2025/2026", pais="ESP", pausa=0.3, debug=False):
    """
    Descarga el Equipo de la Semana de TODAS las jornadas de la temporada
    para una liga, y lo devuelve como un único DataFrame.
    """
    mapa_equipos = mapa_equipos_liga(id_liga, season, pais)

    dfs = []
    for ronda in range(1, num_rondas + 1):
        df_ronda = obtener_totw_jornada(id_liga, ronda, season, mapa_equipos)
        if df_ronda is not None:
            df_ronda["Liga"] = nombre_liga
            df_ronda['url_logo_team'] =  "https://images.fotmob.com/image_resources/logo/teamlogo/"+ df_ronda['teamId'].astype(str) + ".png"
            df_ronda["member_photo"] = "https://images.fotmob.com/image_resources/playerimages/" + df_ronda["player_id"].astype(int).astype(str)+ ".png"
            dfs.append(df_ronda)
            if debug:
                print(f"  ✔ {nombre_liga} jornada {ronda}: {len(df_ronda)} jugadores")
        else:
            if debug:
                print(f"  — {nombre_liga} jornada {ronda}: sin datos (aún no jugada o no disponible)")
        time.sleep(pausa)  # para no machacar la API con requests seguidos

    if not dfs:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)


def totw_global_temporada(df_totw_temporada):
    """
    A partir del DataFrame de todas las jornadas de una liga, calcula
    el 'global' de temporada: cuántas veces ha entrado cada jugador
    en el Equipo de la Semana, con su rating medio.
    """
    if df_totw_temporada.empty:
        return pd.DataFrame()

    resumen = (
        df_totw_temporada
        .assign(rating=pd.to_numeric(df_totw_temporada["rating"], errors="coerce"))
        .groupby(["player_id", "player", "team", "Liga"], as_index=False)
        .agg(veces_en_totw=("Jornada", "count"), rating_medio=("rating", "mean"))
        .sort_values(["veces_en_totw", "rating_medio"], ascending=[False, False])
        .reset_index(drop=True)
    )
    return resumen

Ejecutarlo para las 5 grandes ligas

In [17]:
season_in, season_out = 2025, 2026
temporada = f"{season_in}/{season_out}"

paises_liga = {
    "Premier League": "ENG",
    "LaLiga": "ESP",
    "Serie A": "ITA",
    "Bundesliga": "GER",
    "Ligue 1": "FRA",
}

totw_por_liga = {}
totw_global_por_liga = {}

for nombre_liga, info in ligas_totw.items():
    print(f"\n🏆 {nombre_liga}")
    df_temp = obtener_totw_temporada(
        nombre_liga, info["id"], info["rondas"],
        season=temporada, pais=paises_liga[nombre_liga], debug=True
    )
    totw_por_liga[nombre_liga] = df_temp
    totw_global_por_liga[nombre_liga] = totw_global_temporada(df_temp)


🏆 Premier League
  ✔ Premier League jornada 1: 11 jugadores
  ✔ Premier League jornada 2: 11 jugadores
  ✔ Premier League jornada 3: 11 jugadores
  ✔ Premier League jornada 4: 11 jugadores
  ✔ Premier League jornada 5: 11 jugadores
  ✔ Premier League jornada 6: 11 jugadores
  ✔ Premier League jornada 7: 11 jugadores
  ✔ Premier League jornada 8: 11 jugadores
  ✔ Premier League jornada 9: 11 jugadores
  ✔ Premier League jornada 10: 11 jugadores
  ✔ Premier League jornada 11: 11 jugadores
  ✔ Premier League jornada 12: 11 jugadores
  ✔ Premier League jornada 13: 11 jugadores
  ✔ Premier League jornada 14: 11 jugadores
  ✔ Premier League jornada 15: 11 jugadores
  ✔ Premier League jornada 16: 11 jugadores
  ✔ Premier League jornada 17: 11 jugadores
  ✔ Premier League jornada 18: 11 jugadores
  ✔ Premier League jornada 19: 11 jugadores
  ✔ Premier League jornada 20: 11 jugadores
  ✔ Premier League jornada 21: 11 jugadores
  ✔ Premier League jornada 22: 11 jugadores
  ✔ Premier League jorn

Acceder a los resultados

In [ ]:
# TOTW de una jornada concreta de una liga
totw_por_liga["LaLiga"][totw_por_liga["LaLiga"]["Jornada"] == 37]

# Ranking global de temporada (jugadores con más apariciones en el TOTW)
totw_global_por_liga["LaLiga"].head(15)

# Todas las ligas juntas, global de temporada
df_global_5_ligas = pd.concat(totw_global_por_liga.values(), ignore_index=True)
df_global_5_ligas.sort_values("veces_en_totw", ascending=False).head(20)

,player_id,player,team,Liga,veces_en_totw,rating_medio
0,422685,Bruno Fernandes,Manchester United,Premier League,14,8.692857
634,194165,Harry Kane,Bayern München,Bundesliga,13,9.192308
214,1467236,Lamine Yamal,Barcelona,LaLiga,11,9.100000
635,1029063,Michael Olise,Bayern München,Bundesliga,10,8.850000
430,1347574,Nico Paz,Como,Serie A,10,8.650000
1,737066,Erling Haaland,Manchester City,Premier League,10,9.020000
636,806669,David Raum,RB Leipzig,Bundesliga,9,8.477778
217,1083323,Pedri,Barcelona,LaLiga,9,8.455556
431,605224,Federico Dimarco,Inter,Serie A,9,8.844444
216,517052,Vedat Muriqi,Mallorca,LaLiga,9,8.700000


In [20]:
premier_league= totw_por_liga['Premier League']
premier_league

,player_id,player,rating,rating.bgcolor,teamId,team,pos_x,pos_y,matchId,isTopPlayer,Jornada,Temporada,Liga,url_logo_team,member_photo
0,562727,David Raya,8.6,"rgba(51, 199, 113, 1.0)",9825,Arsenal,0.500,0.100,4813382,None,1,2025/2026,Premier League,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/play...
1,1288450,Rico Lewis,7.9,"rgba(51, 199, 113, 1.0)",8456,Manchester City,0.125,0.357,4813380,None,1,2025/2026,Premier League,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/play...
2,974268,Daniel Ballard,8.7,"rgba(51, 199, 113, 1.0)",8472,Sunderland,0.375,0.357,4813378,None,1,2025/2026,Premier League,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/play...
3,209405,Virgil van Dijk,8.1,"rgba(51, 199, 113, 1.0)",8650,Liverpool,0.625,0.357,4813374,None,1,2025/2026,Premier League,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/play...
4,933845,Rayan Aït-Nouri,8.1,"rgba(51, 199, 113, 1.0)",8456,Manchester City,0.875,0.357,4813380,None,1,2025/2026,Premier League,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/play...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,422685,Bruno Fernandes,8.9,"rgba(51, 199, 113, 1.0)",10260,Manchester United,0.375,0.613,4813745,None,38,2025/2026,Premier League,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/play...
414,524434,João Palhinha,8.9,"rgba(51, 199, 113, 1.0)",8586,Tottenham Hotspur,0.625,0.613,4813753,None,38,2025/2026,Premier League,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/play...
415,1526560,Patrick Dorgu,8.5,"rgba(51, 199, 113, 1.0)",10260,Manchester United,0.875,0.613,4813745,None,38,2025/2026,Premier League,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/play...
416,540088,Ollie Watkins,9.0,"rgba(20, 160, 255, 1.0)",10252,Aston Villa,0.300,0.870,4813750,None,38,2025/2026,Premier League,https://images.fotmob.com/image_resources/logo...,https://images.fotmob.com/image_resources/play...


In [9]:
from IPython.display import HTML, display

def mostrar_campo_con_jugadores(df, escala_posicion=80):

    campo_img = "https://upload.wikimedia.org/wikipedia/commons/thumb/d/d2/Soccer_field_illustration.svg/1200px-Soccer_field_illustration.svg.png"

    html_jugadores = ""

    for _, row in df.iterrows():
        x = row['pos_x'] * escala_posicion
        y = row['pos_y'] * escala_posicion
        foto = row['member_photo']
        logo = row['url_logo_team']
        nombre = row['player']
        rating = row['rating']
        color_fondo = row['rating.bgcolor']

        html_jugadores += f"""
        <div style="
            position:absolute;
            left:{x}%;
            top:{y}%;
            transform:translate(-50%, -50%);
            text-align:center;
            color:white;
            font-weight:bold;
            font-family:Arial,sans-serif;
        ">

            <div style="position:relative; width:35px; height:35px; margin:auto;">

                <img src="{foto}" width="35" height="35"
                     style="border-radius:50%; border:2px solid white;">

                <img src="{logo}" width="15" height="15"
                     style="
                        position:absolute;
                        bottom:-2px;
                        left:30px;
                        border-radius:50%;
                        background:white;
                        padding:1px;
                     ">

            </div>

            <div style="font-size:10px;  margin-top:8px;">{nombre}</div>

            <div style="
                background:{color_fondo};
                color:white;
                margin-top:6px;
                padding:2px 6px;
                border-radius:8px;
                display:inline-block;
                font-size:11px;
            ">
                {rating}
            </div>

        </div>
        """

    html = f"""
    <div style="
        position:relative;
        width:700px;
        height:500px;
        background-image:url('{campo_img}');
        background-size:contain;
        background-repeat:no-repeat;
        background-position:center;
    ">
        {html_jugadores}
    </div>
    """

    display(HTML(html))

In [ ]:
def titulo_team_of_week(selected_round):

    # Visual title
      st.markdown(
         f"""
         <div style="
            background-color:#1e1e1e;
            padding:20px;
            border-radius:12px;
            border:1px solid #333;
            text-align:center;
            margin-bottom:20px;
         ">
            <h1 style="
                  color:#33c771;
                  margin:0;
                  font-size:32px;
            ">
                  ⭐ Team of the Week
            </h1>

            <p style="
                  color:white;
                  font-size:20px;
                  margin:8px 0 0 0;
            ">
                  {selected_round}
            </p>

         </div>
         """,
         unsafe_allow_html=True
      )

In [23]:
jornada = 10
df_jornada = premier_league[premier_league["Jornada"] == jornada]

mostrar_campo_con_jugadores(df_jornada)

In [ ]:
# df_team_of_the_round = create_team_of_the_week_wc26()  
# # Available rounds
# rounds_available = sorted( df_team_of_the_round["round_name"].dropna().unique())

# selected_round = st.selectbox("Select round", rounds_available)

# # Filter selected round
# df_round = df_team_of_the_round[df_team_of_the_round["round_name"] == selected_round].copy()



# titulo_team_of_week(selected_round)
# html = team_of_the_week_plot(df_round)

# components.html(  html,  height=520,scrolling=False)